### Notebook 6: ADK Conversational Agent with Session Persistence

This notebook demonstrates a multi-turn conversational agent where the orchestrator can ask the user for clarification, delegate to specialists, handle errors, and resumewithin the same session. The key difference
from previous notebooks is that the session is reused across calls, so the agent has full conversation history and can ask follow-up questions.

### 1. Imports and Configuration

In [ ]:
import sys
import logging
from pathlib import Path

from google.adk.agents   import Agent
from google.adk.sessions import InMemorySessionService
from google.adk.runners  import Runner
from google.genai         import types
from IPython.display     import display, Markdown

# -----------------------------------------------------------------------
# Path setup
# -----------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / "environment" / ".env")

from config import APP_NAME, USER_ID, strip_emojis
from tools  import (MORTGAGE_TOOLS, RISK_TOOLS, LOAN_TOOLS,
                     INVESTMENT_TOOLS, ALL_TOOLS)

logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt = "%H:%M:%S",
)
logging.getLogger("google.genai").setLevel(logging.ERROR)
log = logging.getLogger(__name__)

MODEL = "gemini-3-flash-preview"

### 2. Agent Setup

The orchestrator has access to all financial tools and three specialist
sub-agents. Its instruction explicitly tells it to ask for clarification
when the user's query is incomplete, rather than guessing or failing.

In [3]:
# -----------------------------------------------------------------------
# Specialists -- same as notebook 3 section 3
# -----------------------------------------------------------------------

mortgage_specialist = Agent(
    model       = MODEL,
    name        = "mortgage_specialist",
    description = "Handles mortgage calculations: payments, amortization, affordability.",
    instruction = (
        "You are a mortgage specialist. Answer the user's mortgage-related "
        "question using the available tools. Be precise with numbers."
    ),
    tools = MORTGAGE_TOOLS,
)

risk_specialist = Agent(
    model       = MODEL,
    name        = "risk_specialist",
    description = "Handles credit risk scoring and loan risk assessment.",
    instruction = (
        "You are a risk analyst. Evaluate the applicant's risk profile "
        "using the available tools. Report the score, category, and any flags."
    ),
    tools = RISK_TOOLS,
)

investment_specialist = Agent(
    model       = MODEL,
    name        = "investment_specialist",
    description = "Handles investment projections, ROI, and savings goals.",
    instruction = (
        "You are an investment analyst. Use the available tools to project "
        "returns, calculate ROI, or plan savings goals."
    ),
    tools = INVESTMENT_TOOLS,
)

# -----------------------------------------------------------------------
# Orchestrator -- instructed to ask for clarification when needed
# -----------------------------------------------------------------------

orchestrator = Agent(
    model       = MODEL,
    name        = "orchestrator",
    description = "Financial services coordinator with conversational ability.",
    instruction = (
        "You are a financial services coordinator. Your job is to help the "
        "user with mortgage, risk, and investment questions by delegating "
        "to the appropriate specialist.\n\n"
        "IMPORTANT RULES:\n"
        "- If the user's query is missing critical information needed for a "
        "calculation (e.g. income, property price, rate, credit score), "
        "ask the user for the missing details before delegating. Do not guess.\n"
        "- If a tool call fails or returns an error, explain the issue to the "
        "user and ask them to provide corrected information.\n"
        "- When you have enough information, delegate to the appropriate "
        "specialist.\n"
        "- After receiving a specialist's response, present it clearly to the "
        "user and ask if they need anything else."
    ),
    sub_agents = [mortgage_specialist, risk_specialist, investment_specialist],
)

### 3. Conversational Loop

The key mechanism: a single `Runner` and `Session` persist across all user messages. Each call to `run_async` appends to the same session history, so the orchestrator sees every previous exchange, enabling:

- Ask a clarifying question and remember the user's answer
- Retry a failed tool call with corrected parameters
- Build on previous results

Type `quit` or `exit` to end the conversation.

In [5]:
# -----------------------------------------------------------------------
# Create a single runner and session that persist across turns.
# This is the critical difference from run_agent() in other notebooks,
# which creates a new session for every call.
# -----------------------------------------------------------------------

session_service = InMemorySessionService()
runner = Runner(
    agent           = orchestrator,
    app_name        = APP_NAME,
    session_service = session_service,
)

session = await runner.session_service.create_session(
    app_name = APP_NAME,
    user_id  = USER_ID,
)

log.info("session created: %s", session.id)
log.info("Type a financial question, or 'quit' to exit.")


06:45:23  INFO      session created: 53db5420-a1b5-4e5d-acdb-6d3ec251a3c5
06:45:23  INFO      Type a financial question, or 'quit' to exit.


In [ ]:
# -----------------------------------------------------------------------
# Multi-turn conversation loop.
#
# This is the core pattern for building conversational agents in ADK.
# Unlike run_agent() in previous notebooks (which creates a fresh
# session per call), this loop reuses a single session across all
# turns. Every user message and agent response is appended to the
# session's event history, which means:
#
#   - The orchestrator sees the full conversation when making decisions
#   - It can reference earlier tool results without re-calling them
#   - It can ask the user a question and act on the answer in the next
#     turn, because the question-answer pair is in the history
#   - Sub-agent responses are also stored, so context accumulates
#
# The loop itself is simple:
#   1. Read user input via input() (blocks until the user types)
#   2. Wrap it as a Content object with role="user"
#   3. Pass it to runner.run_async with the SAME session_id every time
#   4. ADK appends the new message to the session, loads the full
#      history into the LLM context, and runs the agent
#   5. Stream events back -- tool calls, tool responses, and the
#      final text response
#   6. Display the final response and wait for the next input
#
# The session_id is the critical piece. As long as every call uses
# the same session_id, ADK maintains continuity. If you create a new
# session, the orchestrator loses all memory of the conversation.
# -----------------------------------------------------------------------

async def chat_turn(user_text: str):
    """Send one message to the orchestrator and display the response.

    The message is appended to the existing session history. The
    orchestrator receives the full conversation context (all previous
    user messages, tool calls, and agent responses) when generating
    its reply. This is what enables multi-turn clarification flows.
    """

    # Wrap the user's text as an ADK Content object.
    # role="user" marks this as a user turn in the conversation history.
    content = types.Content(
        role  = "user",
        parts = [types.Part(text=user_text)],
    )

    final_text   = None
    final_author = None

    # run_async yields events as the orchestrator processes the message.
    # The session_id is the same one created in the setup cell above,
    # this is what gives the orchestrator memory across turns.
    async for event in runner.run_async(
        user_id     = USER_ID,
        session_id  = session.id,
        new_message = content,
    ):
        author = getattr(event, "author", "unknown")

        # Log tool activity so we can see when the orchestrator
        # delegates to a specialist and what data comes back.
        if event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, "function_call") and part.function_call:
                    log.info("[%s] tool_call: %s(%s)",
                             author, part.function_call.name,
                             dict(part.function_call.args or {}))
                if hasattr(part, "function_response") and part.function_response:
                    log.info("[%s] tool_response: %s -> %s",
                             author, part.function_response.name,
                             str(part.function_response.response)[:200])

        # Collect the final response. This could be:
        # - A clarification question ("What interest rate?")
        # - A specialist's answer after delegation
        # - An error message asking the user to correct something
        # All three are valid final responses, the orchestrator's
        # instruction determines which one it produces.
        if event.is_final_response() and event.content and event.content.parts:
            for part in event.content.parts:
                if not hasattr(part, "text") or not part.text:
                    continue
                if not getattr(part, "thought", False):
                    final_text   = strip_emojis(part.text)
                    final_author = author

    if final_text:
        display(Markdown(f"**{final_author}:**\n\n{final_text}"))

    return final_text


# -----------------------------------------------------------------------
# The input loop. Runs until the user types quit/exit/q.
# Each iteration calls chat_turn() which appends to the same session,
# building up the conversation history one turn at a time.
# -----------------------------------------------------------------------

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit", "q"):
        log.info("conversation ended")
        break
    await chat_turn(user_input)

06:45:54  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
06:45:56  INFO      Response received from the model.


**orchestrator:**

I'd be happy to help you with that! To make sure I connect you with the right specialist, could you tell me a bit more about what you need?

For example:
* Are you looking to calculate monthly payments or see how much house you can afford? (I'll get a **Mortgage Specialist** for you.)
* Are you interested in your credit risk or how likely you are to be approved? (I'll connect you with a **Risk Specialist**.)
* Is this related to a loan for an investment or savings goal? (I'll bring in an **Investment Specialist**.)

Let me know what you're looking for, and I'll get the right expert on the case!

06:46:31  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
06:46:34  INFO      Response received from the model.


**orchestrator:**

To give you an accurate idea of what you can afford, I'll need a few more details to share with our Mortgage Specialist:

1. **Monthly Debts:** Approximately how much do you pay each month for things like car loans, student loans, or credit card minimums?
2. **Down Payment:** How much do you have saved up to put down on a home?
3. **Credit Score:** Do you know your approximate credit score? (This helps us estimate the interest rate you might qualify for.)

Once I have those details, I'll loop in our Mortgage Specialist to run the numbers for you!

06:47:57  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
06:47:58  INFO      Response received from the model.
06:47:58  WARNING   Warning: there are non-text parts in the response: ['function_call'], returning concatenated text result from text parts. Check the full candidates.content.parts accessor to get the full model response.
06:47:58  INFO      [orchestrator] tool_call: transfer_to_agent({'agent_name': 'mortgage_specialist'})
06:47:58  INFO      [orchestrator] tool_response: transfer_to_agent -> {'result': None}
06:47:58  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
06:48:01  INFO      Response received from the model.
06:48:01  INFO      [mortgage_specialist] tool_call: calculate_affordability({'annual_rate': 6.5, 'down_payment': 20000, 'term_years': 30, 'monthly_debts': 100, 'annual_income': 25000})
06:48:01  INFO      [mortgage_specialist] to

**mortgage_specialist:**

Based on your annual income of $25,000, monthly debts of $100, and a down payment of $20,000, here is an estimate of what you can afford:

*   **Maximum Home Price:** ~$145,909
*   **Maximum Loan Amount:** ~$125,909
*   **Estimated Maximum Monthly Mortgage Payment:** ~$796

**Key Details & Assumptions:**
*   **Interest Rate:** Used an estimated 6.5% for a 30-year fixed-rate mortgage.
*   **Debt-to-Income (DTI) Ratio:** This calculation uses a standard 43% DTI limit, meaning your total monthly debt payments (including your new mortgage) would be about $896 per month.
*   **Down Payment:** This includes your $20,000 in savings, which helps lower your monthly payment and increase your total buying power.

Keep in mind that these are estimates. Your final affordability may vary based on actual interest rates, property taxes, homeowners insurance, and any HOA fees. Would you like me to run these numbers again with a different interest rate or loan term?

06:48:30  INFO      conversation ended


### 4. Session Inspection

After the conversation ends, inspect the full session to see every
exchange -- user messages, tool calls, agent responses, and
clarification turns.

In [ ]:
# -----------------------------------------------------------------------
# Retrieve the session and display the full conversation trace.
# -----------------------------------------------------------------------

updated_session = await runner.session_service.get_session(
    app_name   = APP_NAME,
    user_id    = USER_ID,
    session_id = session.id,
)

log.info("total events in session: %d", len(updated_session.events))

for i, ev in enumerate(updated_session.events):
    author = getattr(ev, "author", "?")
    parts_summary = []
    if ev.content and ev.content.parts:
        for p in ev.content.parts:
            if hasattr(p, "function_call") and p.function_call:
                parts_summary.append(f"call:{p.function_call.name}")
            elif hasattr(p, "function_response") and p.function_response:
                parts_summary.append(f"resp:{p.function_response.name}")
            elif hasattr(p, "text") and p.text:
                parts_summary.append(f"text:{p.text[:60]}")
    log.info("  event[%02d] author=%-25s parts=%s", i, author, parts_summary)


### Example Conversation Flow

```
You: I want to know my mortgage payment
Orchestrator: I'd be happy to help. Could you provide the following:
  - Property price
  - Down payment amount
  - Interest rate
  - Loan term (e.g. 30 years)

You: 500k house, 100k down, 6.5%
Orchestrator: Thanks. What loan term: 15 or 30 years?

You: 30 years
Orchestrator: [calls mortgage_specialist -> calculate_monthly_payment]
  Your monthly P&I payment would be $2,528.27...

You: Now check my risk for that loan. I make 130k, have 400/mo debts, 740 credit
Orchestrator: [calls risk_specialist -> assess_loan_risk]
  Your risk level is LOW with a DTI of 28.9%...

You: quit
```